### Import knižníc a príprava pracovného prostredia
V tejto bunke importujeme všetky knižnice potrebné na spracovanie dát, vizualizáciu výsledkov a export výstupov. Používame knižnice `os` na prácu so súbormi a priečinkami, `cv2` na spracovanie obrazu, `numpy` na numerické operácie, `matplotlib` na vizualizáciu, `rasterio` na načítanie rasterových dát a `pandas` na prácu s tabuľkovými výstupmi.

Z knižnice `collections` importujeme triedu `Counter`, ktorú využívame na zistenie rozdelenia objektov do jednotlivých tried. Zároveň importujeme vlastnú triedu `Detector`, ktorá zabezpečuje výpočet príznakov a samotnú klasifikáciu objektov.

V závere bunky nastavujeme predvolenú veľkosť zobrazovaných grafov a vytvárame výstupné priečinky pre vizualizácie a CSV súbory. Tým si pripravujeme pracovné prostredie pre ďalšie spracovanie notebooku.

In [9]:
import os
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import pandas as pd

from detector_my import Detector

plt.rcParams['figure.figsize'] = (10, 10)
os.makedirs('vizualizacie', exist_ok=True)
os.makedirs('QGIS_csv', exist_ok=True)


### Načítanie vstupných dát
V tomto kroku načítavame vstupné dáta pre spracovanie veľkej oblasti. Definujeme cesty k súborom obsahujúcim digitálny výškový model (DEM) vo formáte NPZ a segmentačnú masku vo formáte TIFF.

Najprv inicializujeme inštanciu triedy `Detector`, ktorá bude použitá v ďalších krokoch spracovania. Následne načítavame DEM pomocou knižnice NumPy. V prípade, že súbor obsahuje kľúč `dataset`, použijeme ho priamo, inak načítame prvé dostupné pole.

Segmentačnú masku načítavame pomocou knižnice Rasterio a vyberáme prvý rasterový pás. Masku následne prevádzame na binárnu formu, kde hodnoty väčšie ako nula predstavujú objekty záujmu.

Na záver vypisujeme rozmery DEM a masky, ako aj unikátne hodnoty masky, čím overujeme správnosť načítania vstupných dát.

In [10]:
DEM_NPZ_PATH = 'zdroj/DEM21_opt.npz'
MASK_TIF_PATH = 'zdroj/DEM21_opt_Mounds_Agreg.tif'

detector_my = Detector()

digital_elevation_model_big = np.load(DEM_NPZ_PATH)
if 'dataset' in digital_elevation_model_big:
    dem = digital_elevation_model_big['dataset'].astype(np.float32)
else:
    first_key = list(digital_elevation_model_big.keys())[0]
    dem = digital_elevation_model_big[first_key].astype(np.float32)

with rasterio.open(MASK_TIF_PATH) as src:
    mask = src.read(1)

mask_bin = (mask > 0).astype(np.uint8)

print('DEM shape:', dem.shape)
print('Mask shape:', mask_bin.shape)
print('Mask unique values:', np.unique(mask_bin))


DEM shape: (34995, 22003)
Mask shape: (34995, 22003)
Mask unique values: [0 1]


### Rozdelenie dát na menšie časti
V tomto kroku rozdeľujeme veľkú oblasť na menšie časti (chunky), aby sme mohli spracovanie vykonávať postupne a efektívnejšie. Definujeme funkciu `split_into_chunks_with_coords`, ktorá rozdelí vstupný DEM a masku na pravidelnú mriežku.

Najprv získavame rozmery vstupných dát a pomocou funkcie `np.linspace` určujeme hranice jednotlivých častí v riadkoch a stĺpcoch. Následne prechádzame všetky kombinácie riadkov a stĺpcov a pre každú časť vyrežeme príslušnú oblasť z DEM a masky.

Pre každý chunk ukladáme jeho identifikátor, súradnicové rozsahy a samotné vyrezané dáta do zoznamu `chunk_records`. Tento zoznam následne predstavuje všetky časti, ktoré budeme postupne spracovávať.

Na záver nastavujeme počet riadkov a stĺpcov delenia a vypisujeme počet vytvorených chunkov spolu s informáciami o prvom chunku, čím overujeme správnosť rozdelenia dát.

In [11]:
def split_into_chunks_with_coords(dem, mask, n_rows=10, n_cols=10):
    H, W = dem.shape
    row_edges = np.linspace(0, H, n_rows + 1, dtype=int)
    col_edges = np.linspace(0, W, n_cols + 1, dtype=int)

    chunk_records = []
    for r in range(n_rows):
        for c in range(n_cols):
            row0, row1 = row_edges[r], row_edges[r + 1]
            col0, col1 = col_edges[c], col_edges[c + 1]

            chunk_records.append({
                'chunk_id': len(chunk_records),
                'row0': int(row0),
                'row1': int(row1),
                'col0': int(col0),
                'col1': int(col1),
                'dem': dem[row0:row1, col0:col1],
                'mask': mask[row0:row1, col0:col1],
            })
    return chunk_records

N_ROWS = 10
N_COLS = 10

chunk_records = split_into_chunks_with_coords(dem, mask_bin, n_rows=N_ROWS, n_cols=N_COLS)
print('Počet chunkov:', len(chunk_records))
print('Prvý chunk:', {k: v for k, v in chunk_records[0].items() if k not in ['dem', 'mask']})


Počet chunkov: 100
Prvý chunk: {'chunk_id': 0, 'row0': 0, 'row1': 3499, 'col0': 0, 'col1': 2200}


### Čistenie segmentačnej masky
V tejto bunke definujeme funkciu `clean_mask_for_recall`, ktorá slúži na úpravu segmentačnej masky pred ďalším spracovaním. Cieľom je odstrániť šum a zároveň zachovať čo najviac relevantných objektov.

Najprv prevádzame binárnu masku do 8-bitového formátu v rozsahu 0–255, aby bola kompatibilná s funkciami knižnice OpenCV. Následne aplikujeme morfologickú operáciu otvorenia (opening), ktorá odstraňuje malé izolované objekty a šum.

Potom aplikujeme operáciu uzavretia (closing), ktorá vypĺňa malé medzery a zjednocuje oblasti objektov. Na záver masku prevádzame späť na binárnu formu, ktorú používame v ďalších krokoch spracovania.

In [12]:
def clean_mask_for_recall(mask_bin):
    mask_u8 = (mask_bin * 255).astype(np.uint8)
    opened = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    return (closed > 0).astype(np.uint8)


### Pomocné funkcie na spracovanie kontúr
V tejto bunke definujeme pomocné funkcie, ktoré používame pri spracovaní kontúr a výpočte príznakov objektov vo veľkej oblasti. Tieto funkcie zabezpečujú vytvorenie masky objektu, výpočet výškových charakteristík, zostavenie súboru príznakov a základné filtrovanie kandidátov.

Funkcia `contour_mask` slúži na vytvorenie binárnej masky z jednej kontúry. Vytvoríme prázdne pole zadaného rozmeru a pomocou funkcie `cv2.drawContours` doň vyplníme oblasť kontúry hodnotou 1, čím získame masku konkrétneho objektu.

Funkcia `compute_height_difference` vypočítava výškový rozdiel medzi objektom a jeho bezprostredným okolím. Z masky objektu najprv pomocou dilatácie vytvoríme rozšírenú oblasť a z rozdielu medzi rozšírenou a pôvodnou maskou určujeme prstenec okolo objektu. Následne počítame priemernú výšku objektu a priemernú výšku jeho okolia a vraciame ich rozdiel spolu s oboma priemernými hodnotami.

Funkcia `contour_features` zabezpečuje výpočet hlavných geometrických a výškových príznakov objektu. Z kontúry získavame plochu, obvod, ohraničujúci obdĺžnik, rozmery, pomer strán, konvexný obal, hodnoty solidity a extent a tiež počet vrcholov aproximovaného polygonu. Súčasne vytvárame lokálnu verziu kontúry posunutú do súradnicového systému bounding boxu, z ktorej zostrojíme masku objektu a vypočítame výškové príznaky z príslušného výrezu DEM.

Funkcia `keep_as_candidate` slúži na základné filtrovanie objektov pred samotnou klasifikáciou. Na základe rozmerov, plochy, pomeru strán, hodnôt solidity, extent a výškového rozdielu odstraňujeme nevhodné objekty a na ďalšie spracovanie ponechávame iba relevantných kandidátov.

In [13]:
def contour_mask(shape, contour):
    tmp = np.zeros(shape, dtype=np.uint8)
    cv2.drawContours(tmp, [contour], -1, 1, thickness=-1)
    return tmp


def compute_height_difference(dem_crop, obj_mask, ring_size=3, iterations=2):
    obj_mask_u8 = obj_mask.astype(np.uint8)
    kernel = np.ones((ring_size, ring_size), np.uint8)
    dilated = cv2.dilate(obj_mask_u8, kernel, iterations=iterations)
    ring = ((dilated > 0) & (obj_mask_u8 == 0))

    obj_vals = dem_crop[obj_mask_u8 == 1]
    ring_vals = dem_crop[ring]

    obj_vals = obj_vals[np.isfinite(obj_vals)]
    ring_vals = ring_vals[np.isfinite(ring_vals)]

    if len(obj_vals) == 0:
        return 0.0, np.nan, np.nan

    obj_mean = float(np.mean(obj_vals))
    if len(ring_vals) == 0:
        return 0.0, obj_mean, np.nan

    ring_mean = float(np.mean(ring_vals))
    return float(obj_mean - ring_mean), obj_mean, ring_mean


def contour_features(contour, dem):
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    x, y, w, h = cv2.boundingRect(contour)
    bbox_area = max(w * h, 1)

    rect = cv2.minAreaRect(contour)
    short_side = min(w, h)
    long_side = max(w, h)
    aspect_ratio = long_side / short_side if short_side > 0 else 999.0

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0.0
    extent = area / bbox_area if bbox_area > 0 else 0.0

    epsilon = 0.02 * perimeter if perimeter > 0 else 1.0
    approx = cv2.approxPolyDP(contour, epsilon, True)

    contour_local = contour.copy()
    contour_local[:, 0, 0] -= x
    contour_local[:, 0, 1] -= y

    obj_mask = contour_mask((h, w), contour_local)
    dem_crop = dem[y:y+h, x:x+w]
    height_diff, mean_obj_height, mean_ring_height = compute_height_difference(dem_crop, obj_mask)

    return {
        'contour': contour,
        'contour_local': contour_local,
        'rect': rect,
        'bbox': (x, y, w, h),
        'area': float(area),
        'perimeter': float(perimeter),
        'short_side': float(short_side),
        'long_side': float(long_side),
        'aspect_ratio': float(aspect_ratio),
        'solidity': float(solidity),
        'extent': float(extent),
        'approx_vertices': int(len(approx)),
        'height_diff': float(height_diff),
        'mean_obj_height': float(mean_obj_height) if np.isfinite(mean_obj_height) else np.nan,
        'mean_ring_height': float(mean_ring_height) if np.isfinite(mean_ring_height) else np.nan,
        'obj_mask': obj_mask,
    }


def keep_as_candidate(feat):
    w = feat['bbox'][2]
    h = feat['bbox'][3]
    area = feat['area']
    aspect_ratio = feat['aspect_ratio']
    extent = feat['extent']
    solidity = feat['solidity']
    height_diff = feat['height_diff']

    if w < 3 or h < 3:
        return False
    if area < 30:
        return False
    if aspect_ratio > 12:
        return False
    if solidity < 0.5:
        return False
    if extent < 0.10:
        return False
    if height_diff < -0.5:
        return False
    return True


### Spracovanie chunkov a výber kandidátov
V tomto kroku spracovávame jednotlivé chunky veľkej oblasti a postupne z nich extrahujeme kandidátov na objekty. Inicializujeme zoznam `all_candidate_features`, do ktorého budeme ukladať všetky kandidáty, a zoznam `chunk_stats` na zaznamenanie štatistík pre každý chunk.

Pre každý chunk načítavame jeho DEM, masku a súradnicový posun. Masku najprv čistíme pomocou funkcie `clean_mask_for_recall`, aby sme odstránili šum a zlepšili kvalitu vstupu.

Následne z masky extrahujeme kontúry pomocou funkcie `cv2.findContours`. Pre každú kontúru vypočítavame príznaky pomocou funkcie `contour_features` a pomocou funkcie `keep_as_candidate` vykonávame filtrovanie objektov.

Vybrané kandidáty dopĺňame o informáciu o príslušnom chunke a jeho pozícii v rámci celej oblasti. Tieto kandidáty následne pridávame do globálneho zoznamu `all_candidate_features`.

Pre každý chunk zároveň zaznamenávame počet nájdených kontúr a počet vybraných kandidátov, čo nám umožňuje analyzovať spracovanie jednotlivých častí. Na záver vypisujeme celkový počet kandidátov a vytvárame dátový rámec so štatistikami chunkov.

In [14]:
all_candidate_features = []
chunk_stats = []

for rec in chunk_records:
    chunk_id = rec['chunk_id']
    dem_chunk = rec['dem']
    mask_chunk = rec['mask']
    row0 = rec['row0']
    col0 = rec['col0']

    cleaned_mask = clean_mask_for_recall(mask_chunk)

    contours, _ = cv2.findContours(
        (cleaned_mask * 255).astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    local_candidates = []

    for cnt in contours:
        feat = contour_features(cnt, dem_chunk)
        if keep_as_candidate(feat):
            feat['chunk_id'] = chunk_id
            feat['row0'] = row0
            feat['col0'] = col0
            local_candidates.append(feat)

    all_candidate_features.extend(local_candidates)
    chunk_stats.append({
        'chunk_id': chunk_id,
        'row0': row0,
        'col0': col0,
        'n_contours': len(contours),
        'n_candidates': len(local_candidates),
    })

print('Celkový počet candidate_features:', len(all_candidate_features))
chunk_stats_df = pd.DataFrame(chunk_stats)


Celkový počet candidate_features: 7406


### Príprava vstupov pre klasifikáciu
V tomto kroku pripravujeme vstupné dáta pre klasifikáciu objektov z celej veľkej oblasti. Inicializujeme zoznamy, do ktorých budeme ukladať masky objektov, výškové dáta, rozmery, globálne súradnice a rotované obdĺžniky.

Pre každý kandidátsky objekt získavame jeho lokálne súradnice v rámci chunku a pomocou posunu `row0` a `col0` ich prevádzame na globálne súradnice v rámci celej oblasti. Tým zabezpečujeme správne umiestnenie objektov v globálnom priestore.

Masku objektu preberáme priamo z predpočítaných príznakov a z príslušného chunku vyrezávame zodpovedajúcu časť DEM. Pred uložením kontrolujeme, či majú maska a výškové dáta rovnaké rozmery a nie sú prázdne.

Validné objekty následne ukladáme do zoznamov `house`, `house_height`, `house_size`, `house_cord` a `rectangles_for_angles`, ktoré budú použité v ďalšom kroku klasifikácie.

Na záver vypisujeme počet pripravených objektov, čím overujeme správnosť spracovania.

In [15]:
house = []
house_height = []
house_size = []
house_cord = []
rectangles_for_angles = []

for feat in all_candidate_features:
    x, y, w, h = feat['bbox']
    row0 = feat['row0']
    col0 = feat['col0']
    chunk_id = feat['chunk_id']

    global_x = col0 + x
    global_y = row0 + y

    obj_mask_crop = feat['obj_mask']

    dem_chunk = chunk_records[chunk_id]['dem']
    dem_crop = dem_chunk[y:y+h, x:x+w]

    if obj_mask_crop.size == 0 or dem_crop.size == 0:
        continue
    if obj_mask_crop.shape != dem_crop.shape:
        continue

    house.append(obj_mask_crop.astype(np.uint8))
    house_height.append(dem_crop.astype(np.float32))
    house_size.append([int(w), int(h)])
    house_cord.append([int(global_x), int(global_y), int(w), int(h)])
    rectangles_for_angles.append(feat['rect'])

print('Počet house segmentov:', len(house))


Počet house segmentov: 7406


### Výpočet doplnkových príznakov
Pomocou metódy `compute_difference` vypočítavame pre každý objekt hodnotu rozdielu na základe jeho masky a príslušných výškových dát, ktoré sme uložili do premennej difference.

Pomocou metódy `determine_building_orientations` určujeme orientáciu jednotlivých objektov na základe rotovaných obdĺžnikov. Takto získaná orientácia sa využívala pre určenie budov typu juego.

In [16]:
# Výpočet difference a orientation.
difference = detector_my.compute_difference(house, house_height)
orientation = detector_my.determine_building_orientations(rectangles_for_angles)


c:\Users\kuruc\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\function_base.py:518: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
c:\Users\kuruc\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


### Klasifikácia objektov vo veľkej oblasti
V tomto kroku vykonávame klasifikáciu objektov pre celú veľkú oblasť pomocou triedy `Detector`. Volaním metódy `find_buildings` odovzdávame pripravené vstupy, ako sú masky objektov, ich orientácia, rozmery, globálne súradnice, výškové rozdiely a výškové dáta.

Na rozdiel od spracovania malej oblasti tu ako vstupný obraz nepoužívame vizualizačný podklad, preto do funkcie odovzdávame hodnotu `None`. Klasifikácia je tak vykonaná výhradne na základe vypočítaných príznakov.

Výsledkom je zoznam objektov spolu s ich klasifikáciou. Na záver vypisujeme celkový počet detegovaných objektov a v prípade neprázdneho výsledku aj rozdelenie objektov do jednotlivých tried.

In [17]:
houses = detector_my.find_buildings(
    None,
    house,
    orientation,
    house_size,
    house_cord,
    difference,
    house_height,
)

print('Počet finálnych houses:', len(houses))
if len(houses) > 0:
    print('Rozdelenie tried:', Counter([h[2] for h in houses]))


Vyradený objekt 4465: area=148.0, solidity=0.62, extent=0.41, rect=0.45, diff=-0.21, w=19, h=19
Vyradený objekt 5977: area=171.0, solidity=0.51, extent=0.37, rect=0.33, diff=-0.13, w=26, h=18
Vyradený objekt 6140: area=90.0, solidity=0.62, extent=0.40, rect=0.37, diff=-0.02, w=15, h=15
Vyradený objekt 6614: area=170.0, solidity=0.57, extent=0.42, rect=0.40, diff=-0.22, w=20, h=20
Počet finálnych houses: 6878
Rozdelenie tried: Counter({'01_house': 2306, '16_platform_large': 928, '02_double_house': 754, 'ruin_small': 659, '04_palace': 537, 'cluster_houses': 511, '03_palace': 279, 'unknown_large_building': 253, 'unknown_small_building': 170, '04_palace_large': 137, 'unknown_elongated_building': 123, '15_platform_small': 86, 'ruin_elongated': 58, 'ruin_large_rectangular': 56, '11_temple_pyramida': 8, 'unknown_large_low': 5, 'unknown_small_low': 5, '12_juego': 2, 'unknown_elongated_low': 1})


### Výpočet rotovaného ohraničujúceho obdĺžnika
V tejto bunke definujeme funkciu `get_rotated_box`, ktorá slúži na výpočet vrcholov rotovaného obdĺžnika na základe stredu, rozmerov a uhla natočenia objektu.

Najprv prevádzame uhol zo stupňov na radiány a určujeme polovice strán obdĺžnika. Následne definujeme rohy obdĺžnika v lokálnom súradnicovom systéme vzhľadom na jeho stred.

Pomocou rotačnej matice vykonávame rotáciu týchto bodov o zadaný uhol. Po rotácii body posúvame do globálneho súradnicového systému podľa stredu objektu.

Výsledkom funkcie sú súradnice rotovaného obdĺžnika, ktoré môžeme využiť pri presnejšej vizualizácii v QGISe.

In [18]:
def get_rotated_box(cx, cy, w, h, angle_deg):
    angle = np.deg2rad(angle_deg)

    # polovice strán
    dx = w / 2
    dy = h / 2

    # rohy pred rotáciou (okolo stredu)
    corners = np.array([
        [-dx, -dy],
        [ dx, -dy],
        [ dx,  dy],
        [-dx,  dy]
    ])

    # rotačná matica
    R = np.array([
        [np.cos(angle), -np.sin(angle)],
        [np.sin(angle),  np.cos(angle)]
    ])

    # rotácia + posun do stredu
    rotated = corners @ R.T
    rotated[:, 0] += cx
    rotated[:, 1] += cy

    return rotated

def clean_label(label):
    if label is None:
        return ""
    if "_" in label and label.split("_", 1)[0].isdigit():
        return label.split("_", 1)[1]
    return label

### Export výsledkov do WKT formátu
V tomto kroku realizujeme export výsledkov detekcie a klasifikácie objektov do formátu vhodného pre GIS nástroje. Najprv overujeme, či existujú detegované objekty.

Pre každý objekt získavame jeho stredové súradnice, rozmery a orientáciu. Pomocou funkcie `get_rotated_box` vypočítavame vrcholy rotovaného obdĺžnika, ktorý reprezentuje tvar objektu.

Následne uzatvárame polygon pridaním prvého bodu na koniec a vytvárame textovú reprezentáciu vo formáte WKT (Well-Known Text). Tento formát umožňuje jednoduchý import a vizualizáciu objektov v QGISe.

Výsledné údaje ukladáme do dátového rámca a exportujeme do CSV súboru. Ak nebol detegovaný žiadny objekt, súbor sa nevytvorí a vypíše sa veta, že súbor sa nepodarilo uložiť.


In [19]:
if len(houses) > 0:
    data = []
    data_centers = []

    for h in houses:
        idx = h[7]

        cx = h[0]
        cy = h[1]
        raw_label = h[2]
        label = clean_label(raw_label)
        angle = h[3]
        avg_height = h[4]
        w = h[5]
        h_box = h[6]

        box = get_rotated_box(cx, cy, w, h_box, angle)
        #uzavretenie polygonu (prvý bod na konci)
        coords = list(box) + [box[0]]
        wkt = "POLYGON((" + ", ".join([f"{pt[0]} {pt[1]}" for pt in coords]) + "))"

        data.append({
            'wkt': wkt,
            'label': h[2],
            'orientation': angle,
            'avg_height': h[4],
            'orig_index': idx
        })

        # stredy objektov
        data_centers.append({
            "center_x": cx,
            "center_y": cy,
            "label": label,
            "orientation": angle,
            "avg_height": avg_height,
            "orig_index": idx
        })

    df_wkt = pd.DataFrame(data)
    df_centers = pd.DataFrame(data_centers)

    out_wkt = "QGIS_csv/position_large_wkt.csv"
    out_centers = "QGIS_csv/position_large_centers.csv"

    df_wkt.to_csv(out_wkt, index=False)
    df_centers.to_csv(out_centers, index=False)

    print(f"Uložené do {out_wkt}")
    print(f"Uložené do {out_centers}")
    display(df_wkt.head())
    display(df_centers.head())
else:
    print("Neboli nájdené žiadne houses, súbor sa neuložil.")

Uložené do QGIS_csv/position_large_wkt.csv
Uložené do QGIS_csv/position_large_centers.csv


,wkt,label,orientation,avg_height,orig_index
0,POLYGON((10026.039999559433 3214.2199994236125...,04_palace,8.130102,271.660034,0
1,"POLYGON((10896.5 3061.5, 10896.5 3070.5, 10889...",01_house,90.000000,259.942993,1
2,"POLYGON((10926.0 3037.0, 10926.0 3049.0, 10916...",03_palace,90.000000,258.760223,2
3,"POLYGON((10176.5 2927.0, 10176.5 2937.0, 10163...",16_platform_large,90.000000,256.758026,3
4,"POLYGON((10949.5 2881.0, 10949.5 2893.0, 10942...",01_house,90.000000,259.468506,4


,center_x,center_y,label,orientation,avg_height,orig_index
0,10032,3222,palace,8.130102,271.660034,0
1,10893,3066,house,90.000000,259.942993,1
2,10921,3043,palace,90.000000,258.760223,2
3,10170,2932,platform_large,90.000000,256.758026,3
4,10946,2887,house,90.000000,259.468506,4
